# OpenDART 삼성전자 2019~2023년 재무제표 JSON 가공
이 노트북은 수집된 `data/raw/samsung_electronics_2023_2019.json` 파일을 읽어 `data/processed`에 분석용 CSV/JSON 파일을 생성합니다.

In [ ]:
import json
import csv
import re
from pathlib import Path

RAW_PATH = Path('data/raw/samsung_electronics_2023_2019.json')
PROCESSED_PATH = Path('data/processed')
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

## 원본 JSON 데이터 로드
`data/raw/samsung_electronics_2023_2019.json`을 읽어 전체 데이터를 파이썬 객체로 로드합니다.

In [ ]:
with RAW_PATH.open('r', encoding='utf-8') as f:
    raw_data = json.load(f)

print('meta keys:', list(raw_data.get('meta', {}).keys()))
print('years:', list(raw_data.get('data', {}).keys()))
first_year = next(iter(raw_data.get('data', {})))
print('example item keys:', list(raw_data['data'][first_year][0].keys()))

## 계정명 목록 추출: 연도별, sj_div별
연도와 `sj_div`별로 `account_nm` 목록을 수집하고 중복 없이 정리합니다.

In [ ]:
inventory = []
seen = set()
for year, items in raw_data.get('data', {}).items():
    for item in items:
        row = (str(year), item.get('sj_div', ''), item.get('account_nm', ''))
        if row not in seen:
            seen.add(row)
            inventory.append({
                'year': str(year),
                'sj_div': item.get('sj_div', ''),
                'account_nm': item.get('account_nm', '')
            })

print('inventory count:', len(inventory))
for sample in inventory[:10]:
    print(sample)

## 계정명 목록 CSV 저장
`data/processed/account_name_inventory_2019_2023.csv`로 결과를 저장합니다.

In [ ]:
inventory_path = PROCESSED_PATH / 'account_name_inventory_2019_2023.csv'
with inventory_path.open('w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['year', 'sj_div', 'account_nm'])
    writer.writeheader()
    writer.writerows(inventory)

print('saved inventory to', inventory_path)

## 계정명 가용성 분석
`account_nm`가 각 연도/재무제표구분별로 얼마나 반복되는지 확인합니다.

In [ ]:
availability = {}
for year, items in raw_data.get('data', {}).items():
    for item in items:
        key = (item.get('sj_div', ''), item.get('account_nm', ''))
        availability.setdefault(key, set()).add(str(year))

availability_rows = [
    {'sj_div': key[0], 'account_nm': key[1], 'years_available': ','.join(sorted(years)), 'count_years': len(years)}
    for key, years in availability.items()
]
availability_rows.sort(key=lambda r: (r['sj_div'], r['account_nm']))

print('availability count:', len(availability_rows))
for row in availability_rows[:10]:
    print(row)

In [ ]:
availability_path = PROCESSED_PATH / 'account_availability_2019_2023.csv'
with availability_path.open('w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['sj_div', 'account_nm', 'years_available', 'count_years'])
    writer.writeheader()
    writer.writerows(availability_rows)

print('saved availability to', availability_path)